### Dependencies

In [ ]:
import openai
import pandas as pd  

from IPython.display import Image, display

from qdrant_client import QdrantClient, models
from qdrant_client.models import (
    VectorParams,
    Distance,
    SparseVectorParams,
    Modifier,
    PayloadSchemaType,
    PointStruct,
    Document,
    Prefetch,
    Fusion,
    FusionQuery,
)
from pydantic import BaseModel
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

### Single Node Graph

In [ ]:
# class StateMessage(BaseModel):
#   message: str

# class StateAnswer(BaseModel):
#   answer: str = ""


# class StateVibe(BaseModel):
#   vibe: str

# class State(StateMessage, StateAnswer, StateVibe):
#   pass

class State(BaseModel):
  message: str 
  answer: str = ""
  vibe: str


In [ ]:
from typing import TypedDict


# class AnswerContainer(TypedDict):
#   answer: str

def append_vibes_to_query(state: State) -> State:
  return state.model_copy(update={"answer": f"{state.message} {state.vibe}"})

In [ ]:
workflow = StateGraph(State)
workflow.add_node("append_vibes_to_query", append_vibes_to_query)
workflow.add_edge(START, "append_vibes_to_query")
workflow.add_edge("append_vibes_to_query", END)

graph = workflow.compile()

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
initial_state = State(
  message="Give me some vibes",
  vibe="I'm feeling like a badass today!"
)

In [ ]:
result = graph.invoke(initial_state)

In [ ]:
result

In [ ]:
initial_state = {
  "message": "Give me some vibes",
  "answer": "abc",
  "vibe": "I'm feeling like a badass today!" 
}


In [ ]:
result

### Conditional Graph

In [ ]:
class State(BaseModel):
  message: str
  answer: str = ""

def append_vibes_to_query(state: State) -> State:
  return state.model_copy(update={"answer": state.message})

In [ ]:
def append_vibe_1(state: State) -> State:
  vibe = "I'm feeling like a badass today!"
  return state.model_copy(update={"answer": f"{state.message} {vibe}"})

def append_vibe_2(state: State) -> State:
  vibe = "I'm feeling like a boss today!"
  return state.model_copy(update={"answer": f"{state.message} {vibe}"})

def append_vibe_3(state: State) -> State:
  vibe = "I'm feeling like a legend today!"
  return state.model_copy(update={"answer": f"{state.message} {vibe}"})

In [ ]:
import random
from enum import StrEnum

class Nodes(StrEnum):
    APPEND_VIBES_TO_QUERY = "append_vibes_to_query"
    APPEND_VIBE_1 = "append_vibe_1"
    APPEND_VIBE_2 = "append_vibe_2"
    APPEND_VIBE_3 = "append_vibe_3"


VIBE_OPTIONS = (
    Nodes.APPEND_VIBE_1,
    Nodes.APPEND_VIBE_2,
    Nodes.APPEND_VIBE_3,
)

def router(state: State) -> Nodes:
  return random.choice(list(VIBE_OPTIONS))

workflow = StateGraph(State)
workflow.add_node(Nodes.APPEND_VIBES_TO_QUERY, append_vibes_to_query)
workflow.add_node(Nodes.APPEND_VIBE_1, append_vibe_1)
workflow.add_node(Nodes.APPEND_VIBE_2, append_vibe_2)
workflow.add_node(Nodes.APPEND_VIBE_3, append_vibe_3)

workflow.add_conditional_edges(
  Nodes.APPEND_VIBES_TO_QUERY,
  router,
  {member: member for member in VIBE_OPTIONS},
)

workflow.add_edge(START, Nodes.APPEND_VIBES_TO_QUERY)
workflow.add_edge(Nodes.APPEND_VIBE_1, END)
workflow.add_edge(Nodes.APPEND_VIBE_2, END)
workflow.add_edge(Nodes.APPEND_VIBE_3, END)

graph = workflow.compile()


In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
initial_state = State(
  message="Give me some vibes",
)
result = graph.invoke(initial_state)
result


In [ ]:
def router(state: State) -> Nodes:
    return random.choice(list(VIBE_OPTIONS))


workflow = StateGraph(State)
workflow.add_node(Nodes.APPEND_VIBE_1, append_vibe_1)
workflow.add_node(Nodes.APPEND_VIBE_2, append_vibe_2)
workflow.add_node(Nodes.APPEND_VIBE_3, append_vibe_3)

workflow.add_conditional_edges(
    START,
    router,
    {member: member for member in VIBE_OPTIONS},
)

workflow.add_edge(Nodes.APPEND_VIBE_1, END)
workflow.add_edge(Nodes.APPEND_VIBE_2, END)
workflow.add_edge(Nodes.APPEND_VIBE_3, END)

graph = workflow.compile()


In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
initial_state = State(
    message="Give me some vibes",
)
result = graph.invoke(initial_state)
result


### Exploring LangChain Tool Calling

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

In [ ]:
@tool
def dummy_tool(a: str, b: str) -> str:
  """Concatenate two strings.

  Args:
    a: The first string to concatenate.
    b: The second string to concatenate.

  Returns:
    A string of the two concatenated strings.
  """
  return f"Hello {a} and {b}"

llm = ChatOpenAI(model="gpt-5.4-mini", reasoning_effort="none",
use_responses_api=True)

llm_with_tools = llm.bind_tools([dummy_tool], tool_choice="auto")

response = llm_with_tools.invoke("Use dummy tool to concatenate two random words")


In [ ]:

response

In [ ]:
response.usage_metadata

In [ ]:
response.tool_calls

### Agent Graph

In [ ]:
@tool
def append_vibes(query: str, vibe: str) -> str:
    """Takes in a query and a vibe and returns a string with the query and vibe appended.

    Args:
        query: The query to append the vibe to.
        vibe: The vibe to append to the query.

    Returns:
        A string with the query and vibe appended.
    """

    return f"{query} {vibe}"


In [ ]:
from operator import add
from typing import Annotated, Any, List


class State(BaseModel):
  query: str
  messages: Annotated[List[Any], add] = []
  iteration: int = 0
  answer: str = ""
  final_answer: bool = False

In [ ]:
from jinja2 import Template
from langchain_core.messages import SystemMessage


def agent_node(state: State) -> dict:

    prompt_template = """You are an assistant that is generating vibes for a user.

## Instructions

- You need to use the tools to add vibes to the user's query.
- Add a random vibe to the user's query.

## User Query
{{ query }}
"""

    template = Template(prompt_template)

    prompt = template.render(query=state.query)

    llm = ChatOpenAI(
        model="gpt-5.4-mini", reasoning_effort="none", use_responses_api=True
    )
    llm_with_tools = llm.bind_tools([append_vibes], tool_choice="auto")

    response = llm_with_tools.invoke([SystemMessage(content=prompt)])

    return {"messages": [response]}


In [ ]:
def tool_router(state: State) -> str:
  if (len(state.messages[-1].tool_calls) > 0):
    return "tools"
  else:
    return "end"
    

In [ ]:
workflow = StateGraph(State)

tools = [append_vibes]
tool_node = ToolNode(tools)

workflow.add_edge(START, 'agent_node')
workflow.add_node("tool_node", tool_node)
workflow.add_node("agent_node", agent_node)

workflow.add_conditional_edges(
    "agent_node",
    tool_router,
    {
        "tools":"tool_node", 
        "end": END
    },
)


workflow.add_edge("tool_node", END)

graph = workflow.compile()


In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))